In [58]:
xm_path = '../../data/XM-API/variable_query/2024-05-06_2024-04-29/'
import pandas as pd
cap = pd.read_csv(xm_path+'CapEfecNeta_Recurso.csv')
lis = pd.read_csv(xm_path+'ListadoRecursos_Sistema.csv')
lis = pd.merge(left=lis, right=cap, left_on='Values_Code', right_on='Code', how='inner')
lis = lis.groupby(['Values_Code','Values_Name','Values_Type', 'Values_Disp',
       'Values_RecType', 'Values_EnerSource',
       'Values_OperStartdate', 'Values_State']).agg({
       'Value' : 'mean' }).reset_index()

print('Menores de Agua', lis[(lis['Value'] < 20000) & (lis['Values_EnerSource'] == 'AGUA')].shape)
print('Mayores de Agua', lis[(lis['Value'] > 20000) & (lis['Values_EnerSource'] == 'AGUA')].shape)
print('AGUA', lis[lis['Values_EnerSource'] == 'AGUA'].shape)
lis.to_csv('delete.csv')

minors = lis[(lis['Value'] < 20000) & (lis['Values_EnerSource'] == 'AGUA')]
minors = minors[['Values_Code']]

map = pd.read_csv('../../data/XM-API/Map.csv')
map = pd.merge(left=map, right=minors, on='Values_Code', how='inner')
map = map[['GENERATION_PROJECT']]

geninfo = pd.read_csv('../../model/inputs/gen_infoa.csv')
# Primero creamos una lista de los proyectos de generación que están en 'map'
water_projects_to_update = map['GENERATION_PROJECT'].unique().tolist()

# Luego actualizamos gen_info
geninfo['gen_is_variable'] = geninfo.apply(
    lambda row: 1 if (row['gen_energy_source'] == 'Water' and 
                     row['GENERATION_PROJECT'] in water_projects_to_update) 
               else row['gen_is_variable'], 
    axis=1
)

geninfo.to_csv('../../model/inputs/gen_info.csv', index=False)
print(geninfo['gen_is_variable'].sum())

Menores de Agua (127, 9)
Mayores de Agua (30, 9)
AGUA (157, 9)
199
